# 09 — Context Engineering

**Model:** DeepSeek-V3 via Nebius AI Studio
**Pattern:** Token-Aware Context Compression & Selective Retrieval
**Difficulty:** ⭐⭐⭐⭐☆

---

## The Problem

"Prompt engineering" focuses on *what you say*. **Context engineering** focuses on *what you include* — and what you leave out.

The context window is a scarce resource. Including irrelevant information:
- Distracts the model from the actual task
- Wastes tokens (money and latency)
- In long contexts, causes "lost in the middle" failures

This notebook demonstrates three context engineering techniques:
1. **Context compression**: summarize retrieved documents before including them
2. **Selective routing**: classify the query and only retrieve from the relevant knowledge base
3. **Token budgeting**: enforce a maximum context size and prioritize by relevance score

## Architecture

```
User Query
    │
    ▼
[Query Classifier] ──── determines domain ────▶ [Selective Retrieval]
                                                          │
                                                          ▼
                                               [Context Compressor]
                                                (summarize + score)
                                                          │
                                                          ▼
                                                [Token Budget Filter]
                                                (keep top-k by score)
                                                          │
                                                          ▼
                                                    [LLM Response]
```

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_nebius import ChatNebius
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Optional
from pydantic import BaseModel
from tavily import TavilyClient

llm = ChatNebius(
    model="deepseek-ai/DeepSeek-V3",
    temperature=0,
    api_key=os.environ["NEBIUS_API_KEY"]
)
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
print("Setup complete.")

## Knowledge Domains

In [ ]:
DOMAIN_DOCS = {
    "finance": [
        "Egypt's GDP grew at 3.8% in FY2024, supported by remittances and tourism recovery.",
        "The Egyptian pound stabilized at ~48 EGP/USD after IMF agreement in March 2024.",
        "Egypt's debt-to-GDP ratio stands at approximately 95%, with foreign debt at $165 billion.",
        "Inflation in Egypt peaked at 38% in mid-2023 and declined to ~25% by end of 2024.",
        "The Suez Canal generates approximately $9 billion in annual revenue for Egypt.",
        "Egypt's tourism sector recovered to $13 billion in revenue in 2024.",
    ],
    "technology": [
        "Egypt's technology sector is growing at 12% annually, driven by fintech and e-commerce.",
        "Cairo has emerged as a startup hub with over $1 billion in VC funding in 2023.",
        "Egypt has over 70 million internet users with a 68% penetration rate.",
        "The Egyptian government's Digital Egypt program aims for 95% internet access by 2025.",
        "Key Egyptian unicorns include Fawry, Swvl (now restructured), and MNT-Halan.",
    ],
    "general": [
        "Egypt has a population of approximately 106 million people as of 2024.",
        "Cairo is the capital and largest city with a population of over 20 million.",
        "Arabic is the official language; English is widely spoken in business contexts.",
    ]
}

## State

In [ ]:
class ContextDocument(BaseModel):
    content: str
    relevance_score: float = 0.0
    compressed_content: str = ""
    token_estimate: int = 0


class ContextEngState(TypedDict):
    query: str
    detected_domain: str
    raw_docs: List[str]
    scored_docs: List[ContextDocument]
    filtered_context: str
    token_budget: int
    response: str

## Nodes

In [ ]:
class DomainClassification(BaseModel):
    domain: str
    reasoning: str


domain_classifier = llm.with_structured_output(DomainClassification)


def classify_node(state: ContextEngState) -> dict:
    result: DomainClassification = domain_classifier.invoke([
        SystemMessage(content=(
            "Classify this query into exactly one domain: finance, technology, or general. "
            "Finance: economics, GDP, banking, markets. Technology: software, startups, internet. "
            "General: demographics, culture, geography."
        )),
        HumanMessage(content=f"Query: {state['query']}")
    ])
    domain = result.domain.lower().strip()
    if domain not in DOMAIN_DOCS:
        domain = "general"
    print(f"\n[Classifier] Domain: {domain} — {result.reasoning}")
    return {"detected_domain": domain}


def retrieve_node(state: ContextEngState) -> dict:
    domain = state["detected_domain"]
    docs = DOMAIN_DOCS.get(domain, DOMAIN_DOCS["general"])
    print(f"\n[Retrieve] Fetched {len(docs)} docs from '{domain}' domain.")
    return {"raw_docs": docs}


def compress_and_score_node(state: ContextEngState) -> dict:
    print(f"\n[Compress] Scoring and compressing {len(state['raw_docs'])} documents...")
    scored = []
    for doc in state["raw_docs"]:
        response = llm.invoke([
            SystemMessage(content=(
                "Given a query and a document, output two things separated by '|||':\n"
                "1. A relevance score from 0.0 to 1.0\n"
                "2. A compressed version (1 sentence, query-relevant facts only)\n"
                "Format: SCORE|||COMPRESSED_TEXT\n"
                "Example: 0.85|||Egypt's GDP growth was 3.8% in FY2024."
            )),
            HumanMessage(content=f"Query: {state['query']}\nDocument: {doc}")
        ])
        try:
            parts = response.content.strip().split("|||")
            score = float(parts[0].strip())
            compressed = parts[1].strip() if len(parts) > 1 else doc
        except Exception:
            score = 0.5
            compressed = doc
        token_est = len(compressed) // 4
        scored.append(ContextDocument(
            content=doc, compressed_content=compressed,
            relevance_score=score, token_estimate=token_est
        ))
        print(f"  Score {score:.2f}: {compressed[:80]}...")
    scored.sort(key=lambda d: d.relevance_score, reverse=True)
    return {"scored_docs": scored}


def budget_filter_node(state: ContextEngState) -> dict:
    budget = state["token_budget"]
    used, selected = 0, []
    for doc in state["scored_docs"]:
        if doc.relevance_score < 0.3:
            continue
        if used + doc.token_estimate <= budget:
            selected.append(doc.compressed_content)
            used += doc.token_estimate
    context = "\n".join(f"• {s}" for s in selected)
    print(f"\n[Budget] Selected {len(selected)} docs using ~{used} tokens (budget: {budget}).")
    return {"filtered_context": context}


def respond_node(state: ContextEngState) -> dict:
    print("\n[Respond] Generating answer from engineered context...")
    response = llm.invoke([
        SystemMessage(content=(
            "Answer the question using ONLY the provided context. "
            "If the context doesn't contain enough information, say so clearly."
        )),
        HumanMessage(content=f"Context:\n{state['filtered_context']}\n\nQuestion: {state['query']}")
    ])
    return {"response": response.content}

## Build the Graph

In [ ]:
builder = StateGraph(ContextEngState)
builder.add_node("classify", classify_node)
builder.add_node("retrieve", retrieve_node)
builder.add_node("compress", compress_and_score_node)
builder.add_node("budget_filter", budget_filter_node)
builder.add_node("respond", respond_node)

builder.set_entry_point("classify")
builder.add_edge("classify", "retrieve")
builder.add_edge("retrieve", "compress")
builder.add_edge("compress", "budget_filter")
builder.add_edge("budget_filter", "respond")
builder.add_edge("respond", END)

graph = builder.compile()
print("Context Engineering graph compiled.")

## Live Demo

In [ ]:
queries = [
    "What is Egypt's current economic situation and main challenges?",
    "How is the tech startup ecosystem in Egypt developing?",
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"QUERY: {q}")
    print(f"{'='*60}")
    result = graph.invoke({
        "query": q, "detected_domain": "", "raw_docs": [],
        "scored_docs": [], "filtered_context": "",
        "token_budget": 200, "response": ""
    })
    print(f"\nFINAL ANSWER:\n{result['response']}")

## Key Takeaways

| Technique | Implementation |
|-----------|---------------|
| **Selective retrieval** | Only fetch from the relevant domain |
| **Context compression** | Summarize each doc to 1 sentence |
| **Relevance scoring** | LLM rates 0–1; low scores excluded |
| **Token budgeting** | Greedy packing up to limit |

## What's Next

**Notebook 10** adds a human approval gate — essential for any agent that can take real-world actions.